# Model Optimization: Pruning

In this notebook, we'll apply pruning techniques to our models using distributed processing. Instead of running the pruning on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the pruning on more powerful instances.

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
import os
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from IPython.display import clear_output

## 7. Analyze Pruned Models

Now that the pruning jobs are complete, we'll analyze the pruned models by comparing them to the original models.

In [ ]:
# Import necessary libraries for model loading and evaluation
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM

# Define functions for metrics collection
def get_model_size(model):
    """Calculate model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

def get_num_parameters(model):
    """Calculate number of parameters in the model."""
    return sum(p.numel() for p in model.parameters())

def count_non_zero_params(model):
    """Count non-zero parameters in the model."""
    non_zero = 0
    total = 0
    for param in model.parameters():
        if param.dim() > 1:  # Only count weights, not biases
            non_zero += torch.count_nonzero(param).item()
            total += param.numel()
    return non_zero, total

def measure_inference_time(model, inputs, num_runs=10):
    """Measure average inference time over multiple runs."""
    # Warm-up run
    with torch.no_grad():
        model(**inputs)
    
    # Measure inference time
    start_event = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    end_event = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    
    inference_times = []
    for _ in range(num_runs):
        if torch.cuda.is_available():
            start_event.record()
            with torch.no_grad():
                model(**inputs)
            end_event.record()
            torch.cuda.synchronize()
            inference_times.append(start_event.elapsed_time(end_event))
        else:
            start_time = time.time()
            with torch.no_grad():
                model(**inputs)
            end_time = time.time()
            inference_times.append((end_time - start_time) * 1000)  # Convert to ms
    
    return sum(inference_times) / len(inference_times)

def measure_memory_usage(model, inputs):
    """Measure peak memory usage during inference."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        
        with torch.no_grad():
            model(**inputs)
        
        memory_usage = torch.cuda.max_memory_allocated() / 1024**2  # Convert to MB
    else:
        # For CPU, use a rough estimate based on model size
        memory_usage = get_model_size(model) * 2  # Rough estimate
    
    return memory_usage

def prepare_sample_inputs(model_name, task, tokenizer, device):
    """Prepare sample inputs for the model based on its task."""
    if task == "sequence-classification" or task == "text-classification":
        text = "I really enjoyed this movie. The acting was superb and the plot was engaging."
        inputs = tokenizer(text, return_tensors="pt")
    elif task == "token-classification":
        text = "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington."
        inputs = tokenizer(text, return_tensors="pt")
    elif task == "question-answering":
        question = "What is machine learning?"
        context = "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
        inputs = tokenizer(question, context, return_tensors="pt")
    elif task == "masked-lm" or task == "fill-mask":
        text = "The [MASK] is a large language model trained by OpenAI."
        inputs = tokenizer(text, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Move inputs to the appropriate device
    return {k: v.to(device) for k, v in inputs.items()}

In [ ]:
# Collect metrics for all pruned models
all_metrics = {}

for model_key, job_info in job_configs.items():
    print(f"\nAnalyzing pruned model: {model_key}")
    
    # Get model info
    model_info = model_info_dict[model_key]
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load original model
    print(f"Loading original model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if task == "sequence-classification" or task == "text-classification":
        original_model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        original_model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        original_model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm" or task == "fill-mask":
        original_model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    original_model = original_model.to(device)
    original_model.eval()
    
    # Prepare sample inputs
    inputs = prepare_sample_inputs(model_name, task, tokenizer, device)
    
    # Measure baseline metrics
    baseline_size = get_model_size(original_model)
    baseline_params = get_num_parameters(original_model)
    baseline_non_zero, baseline_total = count_non_zero_params(original_model)
    baseline_inference_time = measure_inference_time(original_model, inputs)
    baseline_memory_usage = measure_memory_usage(original_model, inputs)
    
    print(f"Original model size: {baseline_size:.2f} MB")
    print(f"Original parameters: {baseline_params:,}")
    print(f"Original non-zero weights: {baseline_non_zero:,}/{baseline_total:,} ({baseline_non_zero/baseline_total*100:.2f}%)")
    print(f"Original inference time: {baseline_inference_time:.2f} ms")
    print(f"Original memory usage: {baseline_memory_usage:.2f} MB")
    
    # Load pruned model
    pruned_model_path = os.path.join(job_info["output_path"], f"{model_key}_pruned")
    print(f"Loading pruned model from: {pruned_model_path}")
    
    try:
        if task == "sequence-classification" or task == "text-classification":
            pruned_model = AutoModelForSequenceClassification.from_pretrained(pruned_model_path)
        elif task == "token-classification":
            pruned_model = AutoModelForTokenClassification.from_pretrained(pruned_model_path)
        elif task == "question-answering":
            pruned_model = AutoModelForQuestionAnswering.from_pretrained(pruned_model_path)
        elif task == "masked-lm" or task == "fill-mask":
            pruned_model = AutoModelForMaskedLM.from_pretrained(pruned_model_path)
        
        pruned_model = pruned_model.to(device)
        pruned_model.eval()
        
        # Measure pruned metrics
        pruned_size = get_model_size(pruned_model)
        pruned_params = get_num_parameters(pruned_model)
        pruned_non_zero, pruned_total = count_non_zero_params(pruned_model)
        pruned_inference_time = measure_inference_time(pruned_model, inputs)
        pruned_memory_usage = measure_memory_usage(pruned_model, inputs)
        
        print(f"Pruned model size: {pruned_size:.2f} MB")
        print(f"Pruned parameters: {pruned_params:,}")
        print(f"Pruned non-zero weights: {pruned_non_zero:,}/{pruned_total:,} ({pruned_non_zero/pruned_total*100:.2f}%)")
        print(f"Pruned inference time: {pruned_inference_time:.2f} ms")
        print(f"Pruned memory usage: {pruned_memory_usage:.2f} MB")
        
        # Calculate improvements
        size_reduction = (baseline_size - pruned_size) / baseline_size * 100
        param_reduction = (baseline_params - pruned_params) / baseline_params * 100
        sparsity = (1 - pruned_non_zero / pruned_total) * 100
        time_improvement = (baseline_inference_time - pruned_inference_time) / baseline_inference_time * 100
        memory_reduction = (baseline_memory_usage - pruned_memory_usage) / baseline_memory_usage * 100
        
        print(f"Size reduction: {size_reduction:.2f}%")
        print(f"Parameter reduction: {param_reduction:.2f}%")
        print(f"Model sparsity: {sparsity:.2f}%")
        print(f"Inference time improvement: {time_improvement:.2f}%")
        print(f"Memory usage reduction: {memory_reduction:.2f}%")
        
        # Save metrics
        all_metrics[model_key] = {
            "model_name": model_name,
            "task": task,
            "pruning_method": "structured",
            "pruning_amount": 0.3,
            "model_size": round(pruned_size, 2),
            "inference_time": round(pruned_inference_time, 2),
            "memory_usage": round(pruned_memory_usage, 2),
            "size_reduction": round(size_reduction, 2),
            "time_improvement": round(time_improvement, 2),
            "memory_reduction": round(memory_reduction, 2),
            "parameter_reduction": round(param_reduction, 2),
            "sparsity": round(sparsity, 2),
            "non_zero_weights": pruned_non_zero,
            "total_weights": pruned_total
        }
        
    except Exception as e:
        print(f"Error analyzing pruned model {model_key}: {e}")
        import traceback
        traceback.print_exc()

# Save all metrics to a file
metrics_path = "pruned-metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"\nSaved pruned metrics to {metrics_path}")

## 8. Analyze Model Size Reduction

Now we'll analyze the size reduction achieved through pruning. This analysis helps us understand the impact of pruning on model size and memory footprint.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

for model_key in all_metrics.keys():
    metrics = all_metrics[model_key]
    
    # Prepare data for this model
    model_data = {
        'Model': metrics['model_name'],
        'Size (MB)': metrics['model_size'],
        'Size Reduction (%)': metrics['size_reduction'],
        'Inference Time (ms)': metrics['inference_time'],
        'Time Improvement (%)': metrics['time_improvement'],
        'Sparsity (%)': metrics['sparsity']
    }
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df